# 04 · Corrected label + respiratory features

Notebook 03 found three problems with the team's respiratory-failure label (`scripts/build_respiratory_failure_labels.py`):

| Problem | Evidence from 03 | Fix here |
|---|---|---|
| **High-FiO₂ rule never fires.** It looks for a label called `"fio2"`, but MIMIC calls it *"Inspired O2 Fraction"*; values are also in % (median 40), not fractions. | 0 matching rows | FiO₂ ≥ 60 % (unit-aware) counts as support |
| **Patients already intubated count as "not failed".** Ventilation that started before ICU admission is dropped, and an *Endotracheal tube* on the device chart isn't a criterion. | 12.5 % of the "at-risk" 12 h cohort had an ETT/trach charted before 12 h, with the **same** failure rate as everyone else | Endotracheal/tracheostomy tube, T-piece, a ventilator attached, or ventilation ongoing at admission all count as support |
| **"High flow neb" counts as high-flow oxygen.** It's a humidified aerosol mask, not HFNC. | matched by the keyword `"high flow"` | only *high flow nasal cannula / HFNC / BiPAP / CPAP* |
| **High-flow oxygen can be charted as a flow rate before the device is charted.** | flows > 15 L/min before the landmark in 123–170 at-risk stays (sensitivity check) | O₂ flow > 15 L/min also counts as support (conservative) |

**New label (v2):** *first time on advanced respiratory support* — invasive ventilation, non-invasive ventilation, HFNC (device, or O₂ flow > 15 L/min), or FiO₂ ≥ 60 %.
The landmark question becomes: **among patients not on advanced support by hour L, who starts it between L and 48 h?**

Because "at risk" now means *no advanced support before L*, the respiratory data recorded before L (nasal cannula, oxygen flow ≤ 15 L/min, FiO₂ below 60 %)
is safe to use as features — none of it can already satisfy the label.

The notebook evaluates three set-ups on identical model settings so each change can be seen separately:

| Set-up | Label | Features |
|---|---|---|
| **v1** | team's label (reproduces notebook 02) | labs + medications |
| **v2** | corrected label | labs + medications |
| **v2 + resp** | corrected label | labs + medications + pre-landmark oxygen data |

**Inputs:** same `DATA_DIR` as 02, plus `respiratory_chartevents.csv` and `respiratory_procedureevents.csv`. Reuses the lab cache from 02.
**Outputs:** `landmark_v2_{L}h/train_df.csv, val_df.csv, test_df.csv` (for ClinicalBERT) and `landmark_v2_results.json`.

In [9]:
# ── Setup ─────────────────────────────────────────────────────
import sys, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
pd.options.mode.chained_assignment = None

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DATA_DIR = Path("/content/drive/MyDrive/comb")
SAVE_DIR = Path("/content/drive/MyDrive/USC/ICU-MM-landmark")   # same folder as 02 → reuses its lab cache

LANDMARKS_H = [3, 6, 12]
HORIZON_H   = 48
LAB_START_H = -6
SEED        = 42

TOP_LABS = ["Glucose", "Potassium", "Sodium", "Chloride", "Hemoglobin", "Creatinine", "Urea Nitrogen", "Bicarbonate",
            "Hematocrit", "Anion Gap", "Magnesium", "Platelet Count", "White Blood Cells", "MCHC", "Red Blood Cells",
            "MCV", "MCH", "RDW", "Phosphate", "Calcium, Total"]
DRUG_CATEGORIES = {
    "antibiotic":    ["vancomycin", "piperacillin", "cefepime", "meropenem", "ciprofloxacin", "metronidazole", "levofloxacin", "ampicillin", "ceftriaxone", "azithromycin"],
    "sedation":      ["propofol", "midazolam", "lorazepam", "dexmedetomidine", "ketamine", "diazepam"],
    "opioid":        ["fentanyl", "morphine", "hydromorphone", "dilaudid", "oxycodone", "methadone", "remifentanil"],
    "cardiac":       ["metoprolol", "amiodarone", "digoxin", "diltiazem", "lisinopril", "carvedilol", "atenolol", "labetalol"],
    "anticoagulant": ["heparin", "warfarin", "enoxaparin", "apixaban", "rivaroxaban", "fondaparinux"],
    "insulin":       ["insulin"],
    "diuretic":      ["furosemide", "torsemide", "bumetanide", "spironolactone"],
    "steroid":       ["hydrocortisone", "methylprednisolone", "dexamethasone", "prednisone", "fludrocortisone"],
}
IV_ROUTES, SERIOUS_ROUTE = {"iv", "iv drip", "iv bolus"}, "iv drip"
safe = lambda lab: lab.replace(" ", "_").replace(",", "")

# Respiratory item IDs (as they appear in respiratory_chartevents.csv — see notebook 03)
ITEM_O2_FLOW, ITEM_FIO2, ITEM_VENT_TYPE, ITEM_DEVICE = 223834, 223835, 223848, 226732
HIGH_FLOW_LPM = 15   # nasal cannula tops out ~6 L/min, masks ~15 L/min; more than that counts as high-flow support
SUPPORT_DEVICE = r"high flow nasal cannula|hfnc|bipap|cpap|endotracheal|tracheostomy tube|t-piece"   # counts as advanced support
DEVICE_FEATURES = {                                                                               # lower-level oxygen = features
    "nasal_cannula":   r"nasal cannula",
    "humidified_mask": r"face tent|aerosol|high flow neb|trach mask",
    "high_conc_mask":  r"non-rebreather|venti|face mask|simple mask",
    "oxymizer":        r"oxymizer",
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ── Cohort (same cleaning as 02) ──────────────────────────────
cohort = pd.read_csv(DATA_DIR / "cohort.csv", parse_dates=["icu_intime", "icu_outtime"])
cohort["hadm_id"] = cohort["hadm_id"].astype("int64")
spa = cohort.groupby("hadm_id")["stay_id"].nunique()
multi = cohort[cohort["hadm_id"].isin(spa[spa > 1].index)].sort_values(["hadm_id", "icu_intime"]).copy()
multi["prev_out"] = multi.groupby("hadm_id")["icu_outtime"].shift(1)
bounce = multi.loc[(multi["icu_intime"] - multi["prev_out"]).dt.total_seconds() / 3600 < 24, "stay_id"]
cohort = cohort[~cohort["stay_id"].isin(bounce)].reset_index(drop=True)
cohort["stay_hours"] = (cohort["icu_outtime"] - cohort["icu_intime"]).dt.total_seconds() / 3600

lab_v1 = pd.read_csv(DATA_DIR / "respiratory_failure_labels.csv", parse_dates=["rf_time"])
cohort = cohort.merge(lab_v1[["stay_id", "respiratory_failure", "rf_time"]], on="stay_id", how="left")
cohort["rf_v1_hours"] = np.where(cohort["respiratory_failure"] == 1,
                                 (cohort["rf_time"] - cohort["icu_intime"]).dt.total_seconds() / 3600, np.nan)
print(f"Clean cohort: {len(cohort):,} stays")

Clean cohort: 91,262 stays


In [11]:
# ── Respiratory events ────────────────────────────────────────
intime = cohort.set_index("stay_id")["icu_intime"]
chart = pd.read_csv(DATA_DIR / "respiratory_chartevents.csv", usecols=["stay_id", "charttime", "itemid", "value"], parse_dates=["charttime"])
chart = chart[chart["stay_id"].isin(intime.index)]
chart["hours"] = (chart["charttime"] - chart["stay_id"].map(intime)).dt.total_seconds() / 3600
chart["val_l"] = chart["value"].astype(str).str.lower().str.strip()
chart["num"] = pd.to_numeric(chart["value"], errors="coerce")
fio2_mask = chart["itemid"] == ITEM_FIO2
chart.loc[fio2_mask, "num"] = np.where(chart.loc[fio2_mask, "num"] <= 1, chart.loc[fio2_mask, "num"] * 100, chart.loc[fio2_mask, "num"])  # → percent

proc = pd.read_csv(DATA_DIR / "respiratory_procedureevents.csv", usecols=["stay_id", "starttime", "endtime", "label"], parse_dates=["starttime", "endtime"])
proc = proc[proc["stay_id"].isin(intime.index)]
proc["start_h"] = (proc["starttime"] - proc["stay_id"].map(intime)).dt.total_seconds() / 3600
proc["end_h"]   = (proc["endtime"]   - proc["stay_id"].map(intime)).dt.total_seconds() / 3600

# Every moment a stay is on advanced respiratory support. Support that was already running at admission counts at hour 0.
supp_chart = chart[
    ((chart["itemid"] == ITEM_DEVICE) & chart["val_l"].str.contains(SUPPORT_DEVICE, regex=True)) |
    (chart["itemid"] == ITEM_VENT_TYPE) |
    ((chart["itemid"] == ITEM_FIO2) & (chart["num"] >= 60)) |
    ((chart["itemid"] == ITEM_O2_FLOW) & (chart["num"] > HIGH_FLOW_LPM))       # > 15 L/min is high-flow territory
][["stay_id", "hours"]]
supp_proc = proc[proc["end_h"].isna() | (proc["end_h"] >= 0)][["stay_id", "start_h"]].rename(columns={"start_h": "hours"})
supp = pd.concat([supp_chart, supp_proc])
supp["hours"] = supp["hours"].clip(lower=0)
first_support = supp[supp["hours"] <= HORIZON_H].groupby("stay_id")["hours"].min()
cohort["rf_v2_hours"] = cohort["stay_id"].map(first_support)

for name, col in [("v1 (team label)", "rf_v1_hours"), ("v2 (corrected)", "rf_v2_hours")]:
    h = cohort[col]
    print(f"{name:16s} positive {h.notna().mean()*100:5.1f}% | median onset {h.median():4.1f} h | "
          f"on support within 1 h of admission: {(h <= 1).sum():>6,} ({(h <= 1).mean()*100:4.1f}% of all stays)")
both = pd.crosstab(cohort["rf_v1_hours"].notna(), cohort["rf_v2_hours"].notna(), rownames=["v1 positive"], colnames=["v2 positive"])
print("\nAgreement between labels:"); print(both.to_string())

v1 (team label)  positive  38.2% | median onset  1.9 h | on support within 1 h of admission: 13,351 (14.6% of all stays)
v2 (corrected)   positive  48.2% | median onset  1.1 h | on support within 1 h of admission: 21,355 (23.4% of all stays)

Agreement between labels:
v2 positive  False  True 
v1 positive              
False        47186   9248
True            86  34742


In [12]:
# ── Labs (reuses notebook 02's cache) and prescriptions ───────
MAX_L = max(LANDMARKS_H)
lab_cache = SAVE_DIR / f"_labs_long_upto_{MAX_L}h.parquet"
if lab_cache.exists():
    lab_long = pd.read_parquet(lab_cache)
    print(f"Loaded lab cache from 02: {len(lab_long):,} rows")
else:
    print("No cache from 02 — extracting labs (takes a few minutes)")
    stay_map = cohort[["stay_id", "hadm_id", "icu_intime"]]
    keep_h, keep_l, parts = set(stay_map["hadm_id"]), set(TOP_LABS), []
    for chunk in pd.read_csv(DATA_DIR / "labs.csv", usecols=["hadm_id", "lab_time", "lab_name", "value"], chunksize=2_000_000):
        chunk = chunk[chunk["lab_name"].isin(keep_l) & chunk["hadm_id"].isin(keep_h)].copy()
        if chunk.empty: continue
        chunk["hadm_id"] = chunk["hadm_id"].astype("int64")
        chunk["lab_time"] = pd.to_datetime(chunk["lab_time"])
        chunk["value"] = pd.to_numeric(chunk["value"], errors="coerce")
        chunk = chunk.dropna(subset=["value"]).merge(stay_map, on="hadm_id")
        chunk["hours"] = (chunk["lab_time"] - chunk["icu_intime"]).dt.total_seconds() / 3600
        chunk = chunk[(chunk["hours"] >= LAB_START_H) & (chunk["hours"] <= MAX_L)]
        parts.append(chunk[["stay_id", "lab_name", "hours", "value"]].astype({"hours": "float32", "value": "float32"}))
    lab_long = pd.concat(parts, ignore_index=True)
    lab_long.to_parquet(lab_cache, index=False)

presc = pd.read_csv(DATA_DIR / "prescriptions.csv", usecols=["hadm_id", "med_starttime", "drug", "route"], parse_dates=["med_starttime"])
presc = presc[presc["hadm_id"].isin(set(cohort["hadm_id"]))].copy()
presc["hadm_id"] = presc["hadm_id"].astype("int64")
presc = presc.merge(cohort[["stay_id", "hadm_id", "icu_intime"]], on="hadm_id")
presc["hours"] = (presc["med_starttime"] - presc["icu_intime"]).dt.total_seconds() / 3600
presc = presc[(presc["hours"] >= 0) & (presc["hours"] <= MAX_L)]
presc["drug_l"], presc["route_l"] = presc["drug"].astype(str).str.lower(), presc["route"].astype(str).str.lower()
for cat, kws in DRUG_CATEGORIES.items():
    presc[f"has_{cat}"] = presc["drug_l"].str.contains("|".join(map(re.escape, kws)), regex=True).astype(int)
presc["is_iv"], presc["is_drip"] = presc["route_l"].isin(IV_ROUTES).astype(int), (presc["route_l"] == SERIOUS_ROUTE).astype(int)
print(f"Prescription rows in [0, {MAX_L}h]: {len(presc):,}")

Loaded lab cache from 02: 2,837,949 rows
Prescription rows in [0, 12h]: 2,449,436


In [13]:
# ── Feature builders ──────────────────────────────────────────
LAB_STATS = ["mean", "min", "max", "count", "first", "last", "delta", "early_mean"]

def lab_features(ids, L):
    d = lab_long[(lab_long["hours"] <= L) & lab_long["stay_id"].isin(ids)].sort_values(["stay_id", "lab_name", "hours"])
    agg = d.groupby(["stay_id", "lab_name"])["value"].agg(["mean", "min", "max", "count", "first", "last"])
    agg["delta"] = agg["last"] - agg["first"]
    agg = agg.join(d[(d["hours"] >= 0) & (d["hours"] <= 6)].groupby(["stay_id", "lab_name"])["value"].mean().rename("early_mean"))
    wide = agg.unstack("lab_name")
    wide.columns = [f"{safe(lab)}_{stat}" for stat, lab in wide.columns]
    wide = wide.reindex(index=pd.Index(ids, name="stay_id"), columns=[f"{safe(l)}_{s}" for l in TOP_LABS for s in LAB_STATS])
    cc = [c for c in wide.columns if c.endswith("_count")]
    wide[cc] = wide[cc].fillna(0).astype(int)
    return wide.reset_index()

def presc_features(ids, L):
    d = presc[(presc["hours"] <= L) & presc["stay_id"].isin(ids)]
    g = d.groupby("stay_id")
    out = pd.DataFrame({"total_presc": g.size(), "unique_drugs": g["drug"].nunique(),
                        "iv_count": g["is_iv"].sum(), "iv_drip_count": g["is_drip"].sum()})
    out["iv_drip_ratio"] = out["iv_drip_count"] / out["total_presc"]
    for cat in DRUG_CATEGORIES:
        out[f"has_{cat}"] = g[f"has_{cat}"].max()
    return out.reindex(pd.Index(ids, name="stay_id")).fillna(0).reset_index()

def resp_features(ids, L):
    """Oxygen data charted in [0, L]. For at-risk stays none of it meets the support definition."""
    d = chart[(chart["hours"] >= 0) & (chart["hours"] <= L) & chart["stay_id"].isin(ids)].sort_values(["stay_id", "hours"])
    out = pd.DataFrame(index=pd.Index(ids, name="stay_id"))
    dev = d[d["itemid"] == ITEM_DEVICE]
    for name, pat in DEVICE_FEATURES.items():
        out[f"o2dev_{name}"] = dev[dev["val_l"].str.contains(pat, regex=True)].groupby("stay_id").size().reindex(out.index).gt(0).astype(int)
    out["o2dev_charts"] = dev.groupby("stay_id").size().reindex(out.index).fillna(0)
    for item, prefix in [(ITEM_O2_FLOW, "o2_flow"), (ITEM_FIO2, "fio2")]:
        x = d[(d["itemid"] == item) & d["num"].notna()].groupby("stay_id")["num"]
        out[f"{prefix}_max"], out[f"{prefix}_last"] = x.max(), x.last()
        out[f"{prefix}_count"] = x.size().reindex(out.index).fillna(0)
    out["any_supplemental_o2"] = ((out[[f"o2dev_{k}" for k in DEVICE_FEATURES]].sum(axis=1) > 0) |
                                  (out["o2_flow_max"].fillna(0) > 0)).astype(int)
    return out.reset_index()

In [14]:
# ── Text for ClinicalBERT (02's template + one oxygen sentence) ──
LAB_GROUPS = {
    "Metabolic panel": [("Glucose", "Glucose", "mg/dL"), ("Sodium", "Sodium", "mEq/L"), ("Potassium", "Potassium", "mEq/L"),
                        ("Chloride", "Chloride", "mEq/L"), ("Bicarbonate", "Bicarbonate", "mEq/L"), ("Anion_Gap", "Anion Gap", "mEq/L"),
                        ("Calcium_Total", "Calcium", "mg/dL"), ("Magnesium", "Magnesium", "mg/dL"), ("Phosphate", "Phosphate", "mg/dL")],
    "Renal function":  [("Creatinine", "Creatinine", "mg/dL"), ("Urea_Nitrogen", "BUN", "mg/dL")],
    "Blood count":     [("Hemoglobin", "Hemoglobin", "g/dL"), ("Hematocrit", "Hematocrit", "%"), ("White_Blood_Cells", "WBC", "K/uL"),
                        ("Platelet_Count", "Platelets", "K/uL"), ("Red_Blood_Cells", "RBC", "M/uL"), ("MCHC", "MCHC", "g/dL"),
                        ("MCV", "MCV", "fL"), ("MCH", "MCH", "pg"), ("RDW", "RDW", "%")],
}
CATEGORY_DISPLAY = {"antibiotic": "antibiotics", "sedation": "sedatives", "opioid": "opioids", "cardiac": "cardiac medications",
                    "anticoagulant": "anticoagulation", "insulin": "insulin", "diuretic": "diuretics", "steroid": "corticosteroids"}
DEVICE_DISPLAY = {"nasal_cannula": "nasal cannula", "humidified_mask": "humidified mask", "high_conc_mask": "high-concentration mask", "oxymizer": "oxymizer"}

def row_to_clinical_text(row, with_resp=True):
    parts = [f"Patient is a {int(row['anchor_age'])}-year-old {'male' if row['sex'] == 1 else 'female'}."]
    for group_name, labs in LAB_GROUPS.items():
        mentions = []
        for s, disp, unit in labs:
            mv, dv, cv = row.get(f"{s}_mean"), row.get(f"{s}_delta"), row.get(f"{s}_count", 0)
            if pd.isna(mv) or cv == 0: continue
            m = f"{disp} {mv:.1f} {unit}"
            if not pd.isna(dv) and cv > 1:
                m += " (rising)" if dv > 0 else (" (falling)" if dv < 0 else "")
            mentions.append(m)
        if mentions: parts.append(f"{group_name}: {', '.join(mentions)}.")
    n_presc, n_unique, n_drip = int(row["total_presc"]), int(row["unique_drugs"]), int(row["iv_drip_count"])
    if n_presc == 0:
        parts.append("No medications ordered during observation window.")
    else:
        parts.append(f"Received {n_presc} medication orders ({n_unique} unique drugs) during observation window.")
        if n_drip > 0: parts.append(f"{n_drip} continuous IV {'drip' if n_drip == 1 else 'drips'} running.")
    active = [disp for cat, disp in CATEGORY_DISPLAY.items() if row.get(f"has_{cat}", 0) == 1]
    if active: parts.append(f"Active medications include: {', '.join(active)}.")
    if with_resp:
        devs = [disp for k, disp in DEVICE_DISPLAY.items() if row.get(f"o2dev_{k}", 0) == 1]
        bits = []
        if devs: bits.append(", ".join(devs))
        if not pd.isna(row.get("o2_flow_max")): bits.append(f"oxygen flow up to {row['o2_flow_max']:.0f} L/min")
        if not pd.isna(row.get("fio2_max")): bits.append(f"FiO2 up to {row['fio2_max']:.0f}%")
        parts.append(f"Oxygen: {'; '.join(bits)}." if bits else "No supplemental oxygen documented.")
    return " ".join(parts)

In [15]:
# ── Build + evaluate ──────────────────────────────────────────
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, average_precision_score

def build(L, label_col, with_resp):
    at_risk = (cohort["stay_hours"] > L) & (cohort[label_col].isna() | (cohort[label_col] > L))
    c = cohort[at_risk]
    df = pd.DataFrame({"stay_id": c["stay_id"].values, "subject_id": c["subject_id"].values,
                       "anchor_age": c["anchor_age"].values, "sex": (c["sex"] == "M").astype(int).values,
                       "respiratory_failure": ((c[label_col] > L) & (c[label_col] <= HORIZON_H)).astype(int).values})
    ids = df["stay_id"].tolist()
    df = df.merge(lab_features(ids, L), on="stay_id", how="left").merge(presc_features(ids, L), on="stay_id", how="left")
    if with_resp:
        df = df.merge(resp_features(ids, L), on="stay_id", how="left")
    tr, tmp = next(GroupShuffleSplit(1, test_size=0.30, random_state=SEED).split(df, groups=df["subject_id"]))
    train, tmp = df.iloc[tr].reset_index(drop=True), df.iloc[tmp].reset_index(drop=True)
    va, te = next(GroupShuffleSplit(1, test_size=0.50, random_state=SEED).split(tmp, groups=tmp["subject_id"]))
    val, test = tmp.iloc[va].reset_index(drop=True), tmp.iloc[te].reset_index(drop=True)
    assert not (set(train.subject_id) & set(val.subject_id)) and not (set(train.subject_id) & set(test.subject_id)) \
        and not (set(val.subject_id) & set(test.subject_id)), "patient overlap between splits!"
    return train, val, test

def evaluate(train, val, test, importance=False):
    feats = [c for c in train.columns if c not in {"stay_id", "subject_id", "respiratory_failure", "clinical_text"}]
    models = {"LogReg": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                      LogisticRegression(max_iter=3000, class_weight="balanced")),
              "GradBoost": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, class_weight="balanced", random_state=SEED)}
    out, top = [], None
    for name, m in models.items():
        m.fit(train[feats], train["respiratory_failure"])
        for split, d in [("val", val), ("test", test)]:
            p = m.predict_proba(d[feats])[:, 1]
            out.append({"model": name, "split": split, "AUROC": round(roc_auc_score(d["respiratory_failure"], p), 4),
                        "AUPRC": round(average_precision_score(d["respiratory_failure"], p), 4),
                        "AUPRC_chance": round(d["respiratory_failure"].mean(), 4)})
        if importance and name == "GradBoost":
            pi = permutation_importance(m, val[feats], val["respiratory_failure"], scoring="average_precision",
                                        n_repeats=3, random_state=SEED, n_jobs=-1)
            top = pd.Series(pi.importances_mean, index=feats).sort_values(ascending=False).head(10).round(4)
    return out, top

SETUPS = [("v1", "rf_v1_hours", False), ("v2", "rf_v2_hours", False), ("v2 + resp", "rf_v2_hours", True)]
results, summary_rows = {}, []
for L in LANDMARKS_H:
    results[f"{L}h"] = {}
    for setup, label_col, with_resp in SETUPS:
        t0 = time.time()
        train, val, test = build(L, label_col, with_resp)
        final = setup == "v2 + resp"
        metrics, top = evaluate(train, val, test, importance=final)
        n_pos = int(train.respiratory_failure.sum() + val.respiratory_failure.sum() + test.respiratory_failure.sum())
        results[f"{L}h"][setup] = {"n_train": len(train), "n_val": len(val), "n_test": len(test), "n_positive_total": n_pos,
                                   "positive_rate_train": round(float(train.respiratory_failure.mean()), 4), "metrics": metrics}
        for m in metrics:
            if m["split"] == "test":
                summary_rows.append({"L": f"{L}h", "setup": setup, "model": m["model"], "stays": len(train) + len(val) + len(test),
                                     "positives": n_pos, "AUROC": m["AUROC"], "AUPRC": m["AUPRC"], "chance": m["AUPRC_chance"]})
        print(f"L={L:2d}h  {setup:10s} done ({time.time()-t0:.0f}s)")
        if final:
            results[f"{L}h"][setup]["top10_permutation_importance_val_AUPRC"] = top.to_dict()
            print("   top features (GradBoost, drop in val AUPRC when shuffled):")
            print("   " + top.to_string().replace("\n", "\n   "))
            for d in (train, val, test):
                d["clinical_text"] = d.apply(row_to_clinical_text, axis=1)
            od = SAVE_DIR / f"landmark_v2_{L}h"; od.mkdir(parents=True, exist_ok=True)
            train.to_csv(od / "train_df.csv", index=False); val.to_csv(od / "val_df.csv", index=False); test.to_csv(od / "test_df.csv", index=False)

summary = pd.DataFrame(summary_rows)
print("\n=== Test-set results ===")
print(summary.to_string(index=False))

with open(SAVE_DIR / "landmark_v2_results.json", "w") as f:
    json.dump(results, f, indent=2, default=float)
print(f"\nSaved → {SAVE_DIR / 'landmark_v2_results.json'}  and splits in landmark_v2_*h/")

L= 3h  v1         done (37s)
L= 3h  v2         done (37s)
L= 3h  v2 + resp  done (168s)
   top features (GradBoost, drop in val AUPRC when shuffled):
   o2dev_charts             0.0653
   o2_flow_last             0.0338
   iv_drip_count            0.0183
   Hemoglobin_delta         0.0153
   any_supplemental_o2      0.0141
   Hemoglobin_early_mean    0.0086
   total_presc              0.0056
   Glucose_early_mean       0.0050
   iv_count                 0.0039
   unique_drugs             0.0024
L= 6h  v1         done (35s)
L= 6h  v2         done (31s)
L= 6h  v2 + resp  done (142s)
   top features (GradBoost, drop in val AUPRC when shuffled):
   o2_flow_last           0.0758
   Hemoglobin_count       0.0259
   iv_drip_count          0.0114
   Hemoglobin_delta       0.0112
   o2dev_charts           0.0095
   any_supplemental_o2    0.0084
   o2_flow_count          0.0028
   unique_drugs           0.0024
   Urea_Nitrogen_last     0.0023
   fio2_last              0.0022
L=12h  v1         do